# TG-TX Sort v44 CellBender Workflow

This notebook runs CellBender preprocessing on the TG-TX Sort v44 single-cell dataset.

Goals:
- Prepare raw 10x input files
- Run CellBender remove-background
- Remove ambient RNA contamination
- Save CellBender outputs for downstream Scanpy analysis

Dataset:
- TG-TX-Sort_v44

## 1. Runtime setup

Mount Google Drive and confirm that the Colab GPU is available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

Wed Jan 14 18:39:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
base = "/content/drive/MyDrive/cellbender"
os.chdir(base)
!ls -lh


total 1.9G
drwx------ 2 root root 4.0K Jan  9 16:14 CB_transfer
drwx------ 2 root root 4.0K Jan  9 17:02 CB_transfer2
-rw------- 1 root root 124M Jan  9 17:03 CB_transfer2.tar.gz
-rw------- 1 root root  81M Jan  9 16:35 CB_transfer.tar.gz
drwx------ 2 root root 4.0K Jan  9 19:10 CellBender
-rw------- 1 root root 5.1K Jan  7 20:05 cellbender_test2.ipynb
-rw------- 1 root root 1.3G Jan 14 00:52 ckpt.tar.gz
drwx------ 2 root root 4.0K Jan 11 03:50 results
-rw------- 1 root root  78K Jan 10 05:13 TG_TX_Sort_v44.ipynb
-rw------- 1 root root 165K Jan 14 00:52 TG-TX_v44_cellbender_cell_barcodes.csv
-rw------- 1 root root  23M Jan 14 00:55 TG-TX_v44_cellbender_filtered.h5
-rw------- 1 root root  33M Jan 14 00:55 TG-TX_v44_cellbender.h5
-rw------- 1 root root  23K Jan 14 00:55 TG-TX_v44_cellbender.log
-rw------- 1 root root  581 Jan 14 00:55 TG-TX_v44_cellbender_metrics.csv
-rw------- 1 root root 139K Jan 14 00:52 TG-TX_v44_cellbender.pdf
-rw------- 1 root root 186M Jan 14 00:49 TG-TX_v44_cellb

## 2. Load CellRanger input files

Extract the transferred CellRanger output folder and locate the raw feature-barcode matrix for the sorted TG-TX dataset.

In [ ]:
!tar -xzf CB_transfer2.tar.gz
!ls CB_transfer2
!ls CB_transfer2/TG-TX_v44/outs
!ls CB_transfer2/TG-TX-Sort_v44/outs


cellbender_outputs  TG-TX-Sort_v44  TG-TX_v44
filtered_feature_bc_matrix  raw_feature_bc_matrix
filtered_feature_bc_matrix  raw_feature_bc_matrix


## 3. Prepare raw matrix for CellBender

Copy the raw 10x feature-barcode matrix into the Colab working directory.

In [ ]:
!mkdir -p /content/data/TG-TX-Sort_v44
!ls -ld /content/data/TG-TX-Sort_v44

drwxr-xr-x 2 root root 4096 Jan 14 18:39 /content/data/TG-TX-Sort_v44


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = "/content/drive/MyDrive/cellbender"
os.chdir(base)
!ls -lh


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 1.9G
drwx------ 2 root root 4.0K Jan  9 16:14 CB_transfer
drwx------ 5 root root 4.0K Jan  9 17:02 CB_transfer2
-rw------- 1 root root 124M Jan  9 17:03 CB_transfer2.tar.gz
-rw------- 1 root root  81M Jan  9 16:35 CB_transfer.tar.gz
drwx------ 2 root root 4.0K Jan  9 19:10 CellBender
-rw------- 1 root root 5.1K Jan  7 20:05 cellbender_test2.ipynb
-rw------- 1 root root 1.3G Jan 14 00:52 ckpt.tar.gz
drwx------ 2 root root 4.0K Jan 11 03:50 results
-rw------- 1 root root  78K Jan 10 05:13 TG_TX_Sort_v44.ipynb
-rw------- 1 root root 165K Jan 14 00:52 TG-TX_v44_cellbender_cell_barcodes.csv
-rw------- 1 root root  23M Jan 14 00:55 TG-TX_v44_cellbender_filtered.h5
-rw------- 1 root root  33M Jan 14 00:55 TG-TX_v44_cellbender.h5
-rw------- 1 root root  23K Jan 14 00:55 TG-TX_v44_cellbender.log
-rw------- 1 root root  581 Jan 14 00:55 TG-TX_v44_cellbender_metri

In [ ]:
!tar -xzf CB_transfer2.tar.gz
!ls CB_transfer2

cellbender_outputs  TG-TX-Sort_v44  TG-TX_v44


In [ ]:
!mkdir -p /content/data/TG-TX-Sort_v44_raw
!cp -r /content/drive/MyDrive/cellbender/CB_transfer2/TG-TX-Sort_v44/outs/raw_feature_bc_matrix/* /content/data/TG-TX-Sort_v44_raw/
!ls -lah /content/data/TG-TX-Sort_v44_raw


total 81M
drwxr-xr-x 2 root root 4.0K Jan 14 18:39 .
drwxr-xr-x 4 root root 4.0K Jan 14 18:39 ..
-rw------- 1 root root 4.1M Jan 14 18:39 barcodes.tsv.gz
-rw------- 1 root root 295K Jan 14 18:39 features.tsv.gz
-rw------- 1 root root  77M Jan 14 18:39 matrix.mtx.gz


In [ ]:
!mkdir -p /content/data/TG-TX-Sort_v44_cb
!ls -ld /content/data/TG-TX-Sort_v44_cb

drwxr-xr-x 2 root root 4096 Jan 14 18:39 /content/data/TG-TX-Sort_v44_cb


In [ ]:
import os

BASE = "/content/drive/MyDrive/cellbender/CB_transfer2"

RAW_V44 = f"{BASE}/TG-TX_v44/outs/raw_feature_bc_matrix"
RAW_SORT = f"{BASE}/TG-TX-Sort_v44/outs/raw_feature_bc_matrix"

OUT = f"{BASE}/cellbender_outputs"
os.makedirs(OUT, exist_ok=True)

print(RAW_V44)
print(RAW_SORT)
print(OUT)


/content/drive/MyDrive/cellbender/CB_transfer2/TG-TX_v44/outs/raw_feature_bc_matrix
/content/drive/MyDrive/cellbender/CB_transfer2/TG-TX-Sort_v44/outs/raw_feature_bc_matrix
/content/drive/MyDrive/cellbender/CB_transfer2/cellbender_outputs


## 4. Install CellBender

Install CellBender from the Broad Institute GitHub repository.

In [ ]:

pip install --no-cache-dir -U git+https://github.com/broadinstitute/CellBender.git@4334e8966217c3591bf7c545f31ab979cdc6590d

  Cloning https://github.com/broadinstitute/CellBender.git (to revision 4334e8966217c3591bf7c545f31ab979cdc6590d) to /tmp/pip-req-build-3dt_pckv
  Running command git clone --filter=blob:none --quiet https://github.com/broadinstitute/CellBender.git /tmp/pip-req-build-3dt_pckv
  Running command git rev-parse -q --verify 'sha^4334e8966217c3591bf7c545f31ab979cdc6590d'
  Running command git fetch -q https://github.com/broadinstitute/CellBender.git 4334e8966217c3591bf7c545f31ab979cdc6590d
  Running command git checkout -q 4334e8966217c3591bf7c545f31ab979cdc6590d
  Resolved https://github.com/broadinstitute/CellBender.git to commit 4334e8966217c3591bf7c545f31ab979cdc6590d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 321.3 MB/s 

## 5. Clone CellBender repository

In [ ]:
!git clone https://github.com/broadinstitute/CellBender.git

In [ ]:
%%bash
cp /usr/local/lib/python3.12/dist-packages/cellbender/remove_background/report.py \
   /usr/local/lib/python3.12/dist-packages/cellbender/remove_background/report.py.orig

In [ ]:
%%bash
cp /usr/local/lib/python3.12/dist-packages/cellbender/remove_background/report.py.orig \
   /usr/local/lib/python3.12/dist-packages/cellbender/remove_background/report.py

## 6. Patch CellBender HTML report generation

Apply a compatibility patch to `report.py` to fix boolean-indexing issues encountered during HTML report generation in Google Colab.

In [ ]:
from pathlib import Path
file_path = Path("/usr/local/lib/python3.12/dist-packages/cellbender/remove_background/report.py")
text = file_path.read_text()
text = text.replace(
    "cells = (adata.obs['cell_probability'] > consts.CELL_PROB_CUTOFF)",
    "cells = (adata.obs['cell_probability'] > consts.CELL_PROB_CUTOFF).to_numpy()"
)
text = text.replace(
    "initial_counts = adata.layers[input_layer_key][cells].sum()",
    "initial_counts = adata.layers[input_layer_key][cells.to_numpy()].sum()"
)
text = text.replace(
    "removed_counts = initial_counts - adata.layers[out_key][cells].sum()",
    "removed_counts = initial_counts - adata.layers[out_key][cells.to_numpy()].sum()"
)
text = text.replace(
    "in_counts = np.array(adata.layers[input_layer_key][:, var_logic].sum(axis=1)).squeeze()",
    "in_counts = np.array(adata.layers[input_layer_key][:, var_logic.to_numpy()].sum(axis=1)).squeeze()"
)
text = text.replace(
    "[:, var_logic]",
    "[:, var_logic.to_numpy()]"
)
file_path.write_text(text)
print("Done...")


Done...


In [ ]:
!pwd
!ls -lh ckpt.tar.gz 2>/dev/null || echo "No ckpt.tar.gz in current dir"
!ls -lh /content/drive/MyDrive/cellbender/ckpt.tar.gz 2>/dev/null || echo "No ckpt in Drive cellbender folder"
!find /content -name "ckpt.tar.gz" -type f -maxdepth 6 2>/dev/null
!find /content -name "ckpt.tar.gz" -type f -maxdepth 6 -delete


/content/drive/MyDrive/cellbender
-rw------- 1 root root 1.3G Jan 14 00:52 ckpt.tar.gz
-rw------- 1 root root 1.3G Jan 14 00:52 /content/drive/MyDrive/cellbender/ckpt.tar.gz
/content/drive/MyDrive/cellbender/ckpt.tar.gz
find: warning: you have specified the global option -maxdepth after the argument -name, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.


In [ ]:
!find /content -name "ckpt.tar.gz" -type f -maxdepth 6 2>/dev/null || echo "All ckpt cleared"

In [ ]:
!find /content -name "ckpt.tar.gz" -type f -maxdepth 6 2>/dev/null || echo "All ckpt cleared"
!mkdir -p /content/data/TG-TX-Sort_v44_cb_run1


## 7. Run CellBender remove-background

Run CellBender on the TG-TX Sort v44 dataset using GPU acceleration.

Parameters:
- expected cells: 5000
- total droplets included: 20000
- epochs: 150
- FPR: 0.01

In [ ]:
!cellbender remove-background \
  --input /content/data/TG-TX-Sort_v44_raw \
  --output /content/data/TG-TX-Sort_v44_cb/cellbender_out.h5 \
  --expected-cells 5000 \
  --total-droplets-included 20000 \
  --fpr 0.01 \
  --epochs 150 \
  --cuda


cellbender:remove-background: Command:
cellbender remove-background --input /content/data/TG-TX-Sort_v44_raw --output /content/data/TG-TX-Sort_v44_cb/cellbender_out.h5 --expected-cells 5000 --total-droplets-included 20000 --fpr 0.01 --epochs 150 --cuda
cellbender:remove-background: CellBender 0.3.2
cellbender:remove-background: (Workflow hash 8e69a78883)
cellbender:remove-background: 2026-01-14 19:49:45
cellbender:remove-background: Running remove-background
cellbender:remove-background: Loading data from /content/data/TG-TX-Sort_v44_raw
cellbender:remove-background: CellRanger v3 format
cellbender:remove-background: Features in dataset: 38606 Gene Expression
cellbender:remove-background: Trimming features for inference.
cellbender:remove-background: 33716 features have nonzero counts.
cellbender:remove-background: Prior on counts for cells is 3361
cellbender:remove-background: Prior on counts for empty droplets is 98
cellbender:remove-background: Excluding 7183 features that are estim

## 8. Save CellBender outputs

Copy CellBender outputs back to Google Drive for downstream analysis and long-term storage.

In [ ]:
!mkdir -p /content/data/TG-TX-Sort_v44_raw
!cp -r /content/drive/MyDrive/cellbender/CB_transfer2/TG-TX-Sort_v44/outs/raw_feature_bc_matrix/* /content/data/TG-TX-Sort_v44_raw/
!ls -lah /content/data/TG-TX-Sort_v44_raw

total 81M
drwxr-xr-x 2 root root 4.0K Jan 14 18:39 .
drwxr-xr-x 6 root root 4.0K Jan 14 18:39 ..
-rw------- 1 root root 4.1M Jan 14 19:32 barcodes.tsv.gz
-rw------- 1 root root 295K Jan 14 19:32 features.tsv.gz
-rw------- 1 root root  77M Jan 14 19:32 matrix.mtx.gz


In [ ]:
!mkdir -p /content/drive/MyDrive/cellbender/results/TG-TX-Sort_v44

!cp -r /content/data/TG-TX-Sort_v44_cb/* \
      /content/drive/MyDrive/cellbender/results/TG-TX-Sort_v44/

!ls -lh /content/drive/MyDrive/cellbender/results/TG-TX-Sort_v44


total 190M
-rw------- 1 root root  79K Jan 13 00:14 cellbender_out_cell_barcodes.csv
-rw------- 1 root root  20M Jan 13 00:14 cellbender_out_filtered.h5
-rw------- 1 root root  28M Jan 13 00:11 cellbender_out.h5
-rw------- 1 root root  19K Jan 14 19:32 cellbender_out.log
-rw------- 1 root root  580 Jan 13 00:13 cellbender_out_metrics.csv
-rw------- 1 root root  79K Jan 13 00:12 cellbender_out.pdf
-rw------- 1 root root 143M Jan 13 00:15 cellbender_out_posterior.h5
